# expdpy — Quickstart

A cloud-runnable walkthrough of [expdpy](https://github.com/cmg777/expdpy) on the bundled `kuznets` panel. Run the install cell below first, then run the rest top to bottom.

> If Colab prompts you to **restart the runtime** after the install, do so, then continue from the setup cell.

This notebook mirrors the [Quickstart page](https://cmg777.github.io/expdpy/quickstart.html) of the docs.

In [ ]:
!pip install -q "expdpy[panel] @ git+https://github.com/cmg777/expdpy.git"

In [ ]:
# Ensure Plotly figures render in Google Colab (a no-op in other notebook frontends).
import plotly.io as pio

try:
    import google.colab  # noqa: F401  (present only on Colab)

    pio.renderers.default = "colab"
except ImportError:
    pass

This walkthrough is a guided tour of the whole `expdpy` toolkit — from **describing** a
dataset, through **modelling** it, to **causal panel designs** and **classic panel
estimators**, finishing with the **concept sandboxes** that teach the methods. It uses the
bundled `kuznets` panel (80 countries × 2015–2025) — a synthetic dataset, rich in control
variables, whose regional inequality (`gini_regional`) traces an **N-shaped Kuznets curve**
in income: it rises, falls, then rises again at very high income. The log of GDP per capita
and its square and cube ship as columns (`log_gdp_pc`, `log_gdp_pc_sq`, `log_gdp_pc_cu`) so
the cubic curve is turnkey.

Every figure below is an interactive Plotly chart and every table a styled
[Great Tables](https://posit-dev.github.io/great-tables/) object — both render directly in
Google Colab, Jupyter and the docs you are reading.

In [ ]:
import numpy as np

import expdpy as ex
from expdpy.data import load_kuznets

df = load_kuznets()
df[["country", "continent", "year", "gini_regional", "gdp_pc", "log_gdp_pc"]].head()

## Treating outliers

Winsorize the (skewed) numeric variables at the 1st/99th percentile:

In [ ]:
analysis = ex.treat_outliers(
    df[["gini_regional", "gdp_pc", "school_enrollment", "resource_rents"]],
    percentile=0.01,
).dropna()
analysis.describe().round(3)

## Describe the data

### Descriptive statistics

In [ ]:
ex.prepare_descriptive_table(analysis).gt

### Distributions

`prepare_histogram` summarizes a numeric variable; `prepare_bar_chart` counts the levels of a
categorical one:

In [ ]:
ex.prepare_histogram(df, "gini_regional").fig

In [ ]:
ex.prepare_bar_chart(df, "continent").fig

### Missing values

A heatmap of where data is missing across the panel — essential before you model:

In [ ]:
ex.prepare_missing_values_graph(df, ts_id="year")

## Relationships between variables

### Correlations

Pearson correlations appear above, Spearman below the diagonal; significant cells are bold.

In [ ]:
ex.prepare_correlation_table(analysis).gt

In [ ]:
ex.prepare_correlation_graph(analysis).fig

### Extreme observations

In [ ]:
ex.prepare_ext_obs_table(
    df, n=5, cs_id=["country"], ts_id="year", var="gini_regional"
).gt

## Trends over time

In [ ]:
ex.prepare_trend_graph(df, ts_id="year", var=["gini_regional"]).fig

In [ ]:
ex.prepare_quantile_trend_graph(df, ts_id="year", var="gini_regional").fig

## Comparing groups

In [ ]:
ex.prepare_by_group_bar_graph(df, "continent", "gini_regional", np.nanmedian).fig

In [ ]:
ex.prepare_by_group_violin_graph(df, "continent", "gini_regional")

In [ ]:
ex.prepare_by_group_trend_graph(
    df, ts_id="year", group_var="continent", var="gini_regional"
).fig

## Scatter plot with LOESS

The **N-shaped Kuznets curve** — regional inequality against (log) GDP per capita, sized by
population and colored by continent. The LOESS smoother traces the rise–fall–rise:

In [ ]:
ex.prepare_scatter_plot(
    df, x="log_gdp_pc", y="gini_regional", color="continent", size="population", loess=1
)

## Regression with fixed effects and clustered SEs

`kuznets` is a country–year panel, so the natural specification absorbs **two-way (country +
year) fixed effects** — controlling for every time-invariant country trait and every common
annual shock. A cubic in (log) GDP per capita still recovers the N — a positive, significant
cubic term — *within* country, with standard errors clustered by country:

In [ ]:
res = ex.prepare_regression_table(
    df,
    dvs="gini_regional",
    idvs=["log_gdp_pc", "log_gdp_pc_sq", "log_gdp_pc_cu"],
    feffects=["country", "year"],
    clusters=["country"],
)
res.etable

## Coefficient plot

`prepare_coefficient_plot` turns fitted results into a coefficient-with-confidence-interval
figure — the beginner-friendly alternative to reading a table. Pass several results to
compare specifications side by side; here, pooled OLS against the two-way fixed-effects model:

In [ ]:
pooled = ex.prepare_regression_table(
    df, dvs="gini_regional", idvs=["log_gdp_pc", "log_gdp_pc_sq", "log_gdp_pc_cu"]
)
ex.prepare_coefficient_plot(
    [pooled, res], model_labels=["Pooled OLS", "Two-way FE"]
).fig

## Read the model in plain language

Every result can explain itself. `res.interpret()` gives a plain-language, **associational**
reading of the regression above (never a causal claim unless the design supports it):

In [ ]:
print(res.interpret())

`res.explain()` — or `ex.explain(topic)` — returns a concept explainer that renders as
formatted text, and `ex.list_topics()` lists every available topic:

In [ ]:
ex.explain("fixed_effects")

In [ ]:
ex.list_topics()

## Frisch-Waugh-Lovell plot

The same coefficient, *seen*. `prepare_fwl_plot` residualizes both the outcome and the focal
regressor on the **other** regressors **and** the fixed effects, then scatters the two
residuals. By the Frisch-Waugh-Lovell theorem the fitted slope equals the focal coefficient
in the regression above — here the linear `log_gdp_pc` term, net of the quadratic/cubic terms
and the country and year fixed effects:

In [ ]:
ex.prepare_fwl_plot(
    df,
    dv="gini_regional",
    var="log_gdp_pc",
    controls=["log_gdp_pc_sq", "log_gdp_pc_cu"],
    feffects=["country", "year"],
    clusters=["country"],
).fig

## Post-estimation

Once a model is fitted, `expdpy` exposes the usual post-estimation tools.

**Predictions** — fitted values with the actuals and residuals on the estimation sample:

In [ ]:
ex.prepare_predictions(res).df.head()

**Fixed-effect estimates** — the estimated country intercepts (top 30), the part of the
outcome the panel absorbs:

In [ ]:
ex.prepare_fixef_plot(res, fixef="country").fig

**Joint (Wald) test** — are the curvature terms jointly different from zero? (i.e. is the
relationship non-linear at all?)

In [ ]:
#| warning: false
print(
    ex.prepare_joint_test(res, hypotheses=["log_gdp_pc_sq", "log_gdp_pc_cu"]).summary()
)

## Modern estimators

Beyond OLS, `prepare_estimation` is a single entry point for **IV / 2SLS**, **Poisson** and
**logit / probit** fixed-effects models, richer standard errors (HC, cluster-robust,
Newey–West, Driscoll–Kraay) and stepwise model comparison. Here, instrumental variables:

In [ ]:
ex.prepare_estimation(
    df,
    dv="gini_regional",
    idvs=["log_gdp_pc_sq"],
    model="iv",
    endog=["log_gdp_pc"],
    instruments=["log_gdp_pc_cu"],
    cluster=["country"],
).etable

Switch `model=` to `"poisson"`, `"logit"` or `"probit"` for count and binary outcomes, all
with the same fixed-effects and clustering syntax.

## Treatment structure & event study

For treated panels, the bundled `staggered_did` dataset is a ready-made example — units
adopt a treatment in different years. `prepare_panel_view` shows the **treatment structure**
(who is treated, when):

In [ ]:
from expdpy.data import load_staggered_did

did = load_staggered_did()
ex.prepare_panel_view(did, unit="unit", time="year", cohort="cohort").fig

`prepare_event_study` then traces the **dynamic treatment effect** with a built-in pre-trend
check, using a modern staggered-adoption estimator (Gardner's `did2s` here, with `twfe`,
Sun-Abraham `saturated` and `lpdid` also available):

In [ ]:
ex.prepare_event_study(
    did, outcome="outcome", unit="unit", time="year", cohort="cohort", estimator="did2s"
).fig

## Robust inference

When clusters are few, large-sample cluster-robust p-values can be over-confident.
`prepare_robust_inference` offers **randomization inference** (`ritest`, native to pyfixest)
and the **wild cluster bootstrap** (`wildboot`, needs the optional `wildboottest` package).
Here, randomization inference for the treatment effect, clustering the permutation by unit:

In [ ]:
#| warning: false
m = ex.prepare_regression_table(did, dvs="outcome", idvs=["treated"], clusters=["unit"])
ri = ex.prepare_robust_inference(
    m, param="treated", method="ritest", reps=500, cluster="unit", seed=0
)
print(
    f"Randomization-inference for the treatment effect "
    f"(estimate {ri.estimate:.3f}): p = {ri.p_value:.4f} "
    f"over {ri.reps} permutations."
)

## Classic panel models

With the optional `panel` extra (the Colab notebook installs it for you;
locally `pip install "expdpy[panel]"`), `prepare_panel_table` estimates pooled / between /
fixed / random effects side by side and `prepare_hausman_test` chooses between fixed and
random effects. These run live in the Colab notebook; they are shown here without output
because the documentation build omits the optional dependency.

In [ ]:
#| eval: false
ex.prepare_panel_table(
    did, dv="outcome", idvs=["treated"], entity="unit", time="year"
).etable

In [ ]:
#| eval: false
print(
    ex.prepare_hausman_test(
        did, dv="outcome", idvs=["treated"], entity="unit", time="year"
    ).interpret()
)

## Learn the methods — concept sandboxes

The `sandbox_*` functions simulate data so you can *see* a concept and tune it. Each returns
a figure, a comparison table (`.df`) and a plain-language takeaway (`.interpret()`).

**Omitted-variable bias** — what happens to a coefficient when a correlated confounder is
left out:

In [ ]:
ex.sandbox_omitted_variable_bias(corr_xz=0.6).fig

**Pooled vs fixed effects** — why pooled OLS is biased when the unit effects are correlated
with the regressor (and fixed effects fix it):

In [ ]:
ex.sandbox_pooled_vs_fixed_effects(unit_effect_corr=0.8).fig

**Clustered standard errors** — why ignoring within-cluster correlation understates
uncertainty:

In [ ]:
ex.sandbox_clustering_se(icc=0.3).fig

In [ ]:
print(ex.sandbox_pooled_vs_fixed_effects(unit_effect_corr=0.8).interpret())

## Where to go next

- The [Examples](examples.qmd) page has a basic + advanced snippet for **every** function.
- The [API reference](reference/index.qmd) documents every argument.
- Prefer no code? Launch the [Streamlit](streamlit.qmd) or [Shiny](shiny.qmd) app — the same
  analyses in a point-and-click interface.